# Prepare the RT-ICL datasets
Convert the RepoRT TSV files into descriptor records with retention times in **seconds**. This process needs no API key.

The 10 dataset sources and upstream references are listed in `data/source.tsv` and the per-dataset `*_info.tsv` files. Original RTs are in minutes.

In [1]:
# %pip install -r requirements.txt

In [2]:
from pathlib import Path

ROOT = Path.cwd().resolve()
RAW_DIR = ROOT / "data/raw"
DATA_DIR = ROOT / "data/processed"
RESULTS_DIR = ROOT / "results"

In [3]:
from rt_icl import DEFAULT_DATASETS, preprocess_dataset

DATASET_IDS = list(DEFAULT_DATASETS)
OVERWRITE = False  # Set True only when intentionally rebuilding existing processed files.

# Preprocess

In [4]:
reports = []
for dataset_id in DATASET_IDS:
    result = preprocess_dataset(dataset_id, RAW_DIR, DATA_DIR, overwrite=OVERWRITE)
    report = result.report
    reports.append(report)
    print(f"{dataset_id}: {report['input_rows']} rows -> {report['output_compounds']} compounds; "
          f"{report['excluded_rows']} excluded, {report['merged_rows']} merged")
print("Total compounds:", sum(report["output_compounds"] for report in reports))

0004: 174 rows -> 110 compounds; 56 excluded, 8 merged
0007: 147 rows -> 132 compounds; 15 excluded, 0 merged
0027: 303 rows -> 192 compounds; 92 excluded, 19 merged
0178: 88 rows -> 81 compounds; 7 excluded, 0 merged
0182: 207 rows -> 186 compounds; 20 excluded, 1 merged
0231: 137 rows -> 125 compounds; 11 excluded, 1 merged
0234: 132 rows -> 132 compounds; 0 excluded, 0 merged
0235: 136 rows -> 133 compounds; 2 excluded, 1 merged
0283: 74 rows -> 72 compounds; 2 excluded, 0 merged
0317: 172 rows -> 162 compounds; 7 excluded, 3 merged
Total compounds: 1325


## Inspect the processed data

In [5]:
from rt_icl import load_dataset

example = load_dataset(DATASET_IDS[0], DATA_DIR)
display(example.compounds[:2])
display(example.lc_condition)
display(reports[0])

[{'SMILES': 'CC(C)(O)CC(=O)O',
  'Mol_Formula': 'C5H10O3',
  'Exact_Mass': 118.06,
  'LogP': 0.23,
  'TPSA': 57.53,
  'HBA_Count': 2,
  'HBD_Count': 2,
  'Aromatic_Ring_Count': 0,
  'Name': '3-hydroxy-3-methylbutyric acid (3-hydroxyisovaleric acid) ',
  'RT': 252.0},
 {'SMILES': 'C[N+](C)(C)[O-]',
  'Mol_Formula': 'C3H9NO',
  'Exact_Mass': 75.07,
  'LogP': 0.19,
  'TPSA': 23.06,
  'HBA_Count': 1,
  'HBD_Count': 0,
  'Aromatic_Ring_Count': 0,
  'Name': 'trimethylamine oxide ',
  'RT': 54.0}]

{'dataset_id': '0004',
 'column_type': 'Reversed-Phase',
 'column': 'Thermo Scientific Hypersil GOLD (2.1 mm × 150 mm, 1.9 μm)',
 'mobile_phase_A': '0.1% formic acid in water (pH 3)',
 'mobile_phase_B': '0.1% formic acid in acetonitrile (pH 3)',
 'flow_rate': '0.5 mL/min',
 'column_temperature': 'NA',
 'gradient': ['0 min (0 s): 100% A, 0% B',
  '2 min (120 s): 100% A, 0% B',
  '13 min (780 s): 0% A, 100% B',
  '15.5 min (930 s): 0% A, 100% B',
  '19 min (1140 s): 100% A, 0% B']}

{'input_rows': 174,
 'output_compounds': 110,
 'excluded_rows': 56,
 'merged_rows': 8,
 'exclusions': [{'row': 5,
   'id': '0004_00005',
   'reason': 'duplicate_rt_spread_gt_0.1_min'},
  {'row': 6, 'id': '0004_00006', 'reason': 'duplicate_rt_spread_gt_0.1_min'},
  {'row': 7, 'id': '0004_00007', 'reason': 'duplicate_rt_spread_gt_0.1_min'},
  {'row': 8, 'id': '0004_00008', 'reason': 'duplicate_rt_spread_gt_0.1_min'},
  {'row': 10, 'id': '0004_00010', 'reason': 'duplicate_rt_spread_gt_0.1_min'},
  {'row': 23, 'id': '0004_00023', 'reason': 'duplicate_rt_spread_gt_0.1_min'},
  {'row': 25, 'id': '0004_00025', 'reason': 'duplicate_rt_spread_gt_0.1_min'},
  {'row': 26, 'id': '0004_00026', 'reason': 'duplicate_rt_spread_gt_0.1_min'},
  {'row': 35, 'id': '0004_00035', 'reason': 'duplicate_rt_spread_gt_0.1_min'},
  {'row': 37, 'id': '0004_00037', 'reason': 'duplicate_rt_spread_gt_0.1_min'},
  {'row': 38, 'id': '0004_00038', 'reason': 'duplicate_rt_spread_gt_0.1_min'},
  {'row': 49, 'id': '0004_00

Outputs are written to `data/processed/<dataset-id>/`: attributes JSON, LC condition YAML, and a preprocessing report. The bundled inputs produce 1,325 compounds in total.

The same operation is available from a terminal: `python -m rt_icl.preprocessing`.